# How has the distribution of monthly IPCA inflation changed?

This notebook studies **realized monthly IPCA inflation**, calculated from the IBGE national price index. It compares three equal eight-year periods—2002–2009, 2010–2017, and 2018–2025—and tracks the distribution through rolling 36-month windows. Each period contains every calendar month eight times, making the histogram comparisons less sensitive to seasonality.

The analysis is descriptive: histograms estimate density, empirical CDFs compare probabilities, and rolling quantiles show how the center and tails move. Data source: [IBGE SIDRA table 1737, variable 2266](https://sidra.ibge.gov.br/tabela/1737). The [local data instructions](data/README.md) explain how to rebuild the CSV.


## 1. Calculate monthly inflation from the IPCA index

Use the month-to-month percentage change in the index. The comparison covers January 2002 through December 2025, giving 288 monthly rates. Run from the repository root or projects/.


In [ ]:
index_file <- file.path("projects", "data", "ibge_ipca_monthly_index_200110_202512.csv")
project_root <- if (file.exists(index_file)) {
  "."
} else if (file.exists(file.path("..", index_file))) {
  ".."
} else {
  stop("Missing IBGE index; run projects/download_ibge_ipca.R first")
}
ibge <- read.csv(file.path(project_root, index_file), colClasses = c("character", "numeric"))
expected_months <- format(seq(as.Date("2001-10-01"), as.Date("2025-12-01"), by = "month"), "%Y%m")
stopifnot(identical(ibge$month, expected_months),
          all(is.finite(ibge$ipca_index)), all(ibge$ipca_index > 0))

rate_months <- ibge$month[-1L]
monthly <- data.frame(
  month = rate_months,
  date = as.Date(paste0(substr(rate_months, 1, 4), "-",
                        substr(rate_months, 5, 6), "-01")),
  ipca_pct = 100 * (ibge$ipca_index[-1L] / ibge$ipca_index[-nrow(ibge)] - 1)
)
monthly <- monthly[monthly$month >= "200201", , drop = FALSE]
row.names(monthly) <- NULL
stopifnot(nrow(monthly) == 288L, all(is.finite(monthly$ipca_pct)))
cat("Monthly observations:", nrow(monthly), "|", monthly$month[1], "to",
    tail(monthly$month, 1), "\n")


## 2. Describe three equal historical periods

Use the same observations, units, and calendar-month composition in each period. Compare mean, standard deviation, median, 10th and 90th percentiles, and the observed frequency of deflation or inflation above 1% in a month.


In [ ]:
period_labels <- c("2002–2009", "2010–2017", "2018–2025")
year <- as.integer(substr(monthly$month, 1, 4))
monthly$period <- cut(year, breaks = c(2001, 2009, 2017, 2025),
                      labels = period_labels)
stopifnot(identical(as.integer(table(monthly$period)), rep(96L, 3L)))
values_by_period <- split(monthly$ipca_pct, monthly$period)
distribution_summary <- do.call(rbind, lapply(period_labels, function(period) {
  x <- values_by_period[[period]]
  data.frame(period = period, months = length(x),
             mean_pct = mean(x), sd_pct = sd(x), median_pct = median(x),
             p10_pct = unname(quantile(x, 0.10)),
             p90_pct = unname(quantile(x, 0.90)),
             negative_share_pct = 100 * mean(x < 0),
             above_1_share_pct = 100 * mean(x > 1))
}))
print(distribution_summary, row.names = FALSE, digits = 4)


**Validated local results (96 months per period):**

| Period | Mean monthly IPCA | SD | Median | Negative months | Months above 1% |
| --- | ---: | ---: | ---: | ---: | ---: |
| 2002–2009 | 0.533% | 0.454 p.p. | 0.445% | 3 | 7 |
| 2010–2017 | 0.510% | 0.298 p.p. | 0.505% | 1 | 5 |
| 2018–2025 | 0.428% | 0.410 p.p. | 0.385% | 11 | 9 |

These summaries show changes in both center and spread; they do not by themselves identify why the changes occurred.


## 3. Compare histograms using identical bins

All three panels use the same 0.2 percentage-point bin width and axis limits. The y-axis is density, so the shapes remain comparable even if the period lengths change later. The dashed line marks each period's median.


In [ ]:
period_colors <- c("steelblue4", "darkorange2", "darkgreen")
common_breaks <- seq(floor(min(monthly$ipca_pct) * 5) / 5,
                     ceiling(max(monthly$ipca_pct) * 5) / 5, by = 0.2)
hist_data <- lapply(values_by_period, hist, breaks = common_breaks, plot = FALSE)
density_max <- max(vapply(hist_data, function(h) max(h$density), numeric(1)))

options(repr.plot.width = 12, repr.plot.height = 4.5)
old_par <- par(no.readonly = TRUE)
par(bg = "white", mfrow = c(1, 3), mar = c(4.5, 4, 3, 1))
for (i in seq_along(period_labels)) {
  x <- values_by_period[[period_labels[i]]]
  hist(x, breaks = common_breaks, freq = FALSE,
       xlim = range(common_breaks), ylim = c(0, density_max * 1.12),
       col = period_colors[i], border = "white",
       xlab = "Monthly IPCA (%)", ylab = "Density",
       main = period_labels[i])
  abline(v = median(x), col = "black", lty = 2, lwd = 1.5)
}
par(old_par)


![Monthly IPCA histograms for three equal historical periods](figures/ipca_monthly_distribution_histograms.png)


## 4. Compare empirical CDFs

At any x, the ECDF gives the share of months with inflation at or below x. The table reports the largest vertical gap between each pair of ECDFs (the descriptive Kolmogorov–Smirnov distance). It does not assign a p-value, because monthly rates can be serially dependent.


In [ ]:
options(repr.plot.width = 10, repr.plot.height = 6)
old_par <- par(no.readonly = TRUE)
par(bg = "white")
plot(ecdf(values_by_period[[1]]), do.points = FALSE, verticals = TRUE,
     xlim = range(common_breaks), col = period_colors[1], lwd = 2,
     xlab = "Monthly IPCA (%)", ylab = "Share of months at or below x",
     main = "Monthly IPCA distributions across periods")
for (i in 2:3) {
  lines(ecdf(values_by_period[[i]]), do.points = FALSE, verticals = TRUE,
        col = period_colors[i], lwd = 2)
}
legend("bottomright", period_labels, col = period_colors, lty = 1, lwd = 2, bty = "n")
par(old_par)

grid <- sort(unique(monthly$ipca_pct))
pairs <- combn(seq_along(period_labels), 2, simplify = FALSE)
ks_distances <- do.call(rbind, lapply(pairs, function(pair) {
  a <- values_by_period[[pair[1]]]
  b <- values_by_period[[pair[2]]]
  data.frame(first_period = period_labels[pair[1]],
             second_period = period_labels[pair[2]],
             ecdf_distance = max(abs(ecdf(a)(grid) - ecdf(b)(grid))))
}))
print(ks_distances, row.names = FALSE, digits = 4)


![Empirical CDFs of monthly IPCA across three historical periods](figures/ipca_monthly_distribution_ecdfs.png)

The largest ECDF gap is **0.198** between 2010–2017 and 2018–2025. This is a descriptive distance, without an independence-based significance claim.


## 5. Track the distribution with rolling 36-month windows

Each window contains three complete calendar years. Plot the 10th to 90th percentile band and median of monthly inflation, then the standard deviation. Rolling windows overlap, so this is a description of change over time rather than independent estimates.


In [ ]:
window_months <- 36L
window_ends <- seq.int(window_months, nrow(monthly))
rolling_values <- t(vapply(window_ends, function(last_month) {
  x <- monthly$ipca_pct[(last_month - window_months + 1L):last_month]
  c(p10 = unname(quantile(x, 0.10)), median = median(x),
    p90 = unname(quantile(x, 0.90)), sd = sd(x))
}, numeric(4)))
rolling <- data.frame(date = monthly$date[window_ends], rolling_values)
stopifnot(nrow(rolling) == 253L, all(is.finite(as.matrix(rolling[-1]))))

options(repr.plot.width = 11, repr.plot.height = 7)
old_par <- par(no.readonly = TRUE)
par(bg = "white", mfrow = c(2, 1), mar = c(3, 4.5, 3, 1))
plot(rolling$date, rolling$median, type = "n",
     ylim = range(c(rolling$p10, rolling$p90)),
     xlab = "", ylab = "Monthly IPCA (%)",
     main = "Rolling 36-month distribution: center and tails")
polygon(c(rolling$date, rev(rolling$date)),
        c(rolling$p10, rev(rolling$p90)),
        col = rgb(0.2, 0.45, 0.7, 0.25), border = NA)
lines(rolling$date, rolling$median, col = "steelblue4", lwd = 2)
abline(h = 0, col = "grey50", lty = 3)
legend("topleft", c("Median", "10th–90th percentile band"),
       col = c("steelblue4", rgb(0.2, 0.45, 0.7, 0.25)),
       lty = c(1, NA), lwd = c(2, NA), pch = c(NA, 15), bty = "n")

par(mar = c(4.5, 4.5, 3, 1))
plot(rolling$date, rolling$sd, type = "l", col = "darkorange3", lwd = 2,
     xlab = "End of 36-month window", ylab = "Standard deviation (p.p.)",
     main = "Rolling 36-month dispersion")
par(old_par)


![Rolling 36-month median, percentile band, and standard deviation of monthly IPCA](figures/ipca_monthly_distribution_rolling.png)


## 6. Check the period contrast by permuting whole years

Use the **largest pairwise ECDF gap** as one summary of how separated the three periods are. Under a reference scenario in which the 24 complete years can be reassigned among three groups of eight, shuffle whole years 10,000 times. This preserves each year's monthly pattern but assumes years are exchangeable and does not preserve dependence across years. The resulting tail fraction is an approximate diagnostic, not proof of stationarity.


In [ ]:
year_blocks <- split(monthly$ipca_pct, year)
stopifnot(length(year_blocks) == 24L, all(lengths(year_blocks) == 12L))
max_gap <- function(groups) {
  curves <- lapply(groups, function(x) ecdf(x)(grid))
  max(abs(curves[[1]] - curves[[2]]),
      abs(curves[[1]] - curves[[3]]),
      abs(curves[[2]] - curves[[3]]))
}
observed_gap <- max(ks_distances$ecdf_distance)
set.seed(20261110)
B <- 10000L
permuted_gaps <- numeric(B)
for (b in seq_len(B)) {
  shuffled_years <- sample.int(length(year_blocks))
  groups <- lapply(split(shuffled_years, rep(1:3, each = 8L)), function(ids) {
    unlist(year_blocks[ids], use.names = FALSE)
  })
  permuted_gaps[b] <- max_gap(groups)
}
stopifnot(all(is.finite(permuted_gaps)))
permutation_result <- data.frame(
  observed_max_gap = observed_gap,
  median_shuffled_gap = median(permuted_gaps),
  shuffled_95th_percentile = unname(quantile(permuted_gaps, 0.95)),
  fraction_at_least_observed = (1 + sum(permuted_gaps >= observed_gap)) / (B + 1)
)
print(permutation_result, row.names = FALSE, digits = 4)

options(repr.plot.width = 10, repr.plot.height = 6)
par(bg = 'white')
hist(permuted_gaps, breaks = 30, col = 'lightblue', border = 'white',
     xlab = 'Largest pairwise ECDF gap after shuffling years',
     main = 'Year-block permutation reference distribution')
abline(v = observed_gap, col = 'navy', lwd = 2)
abline(v = quantile(permuted_gaps, 0.95), col = 'red', lty = 2, lwd = 2)
legend('topright', c('Observed gap', '95th percentile of shuffled gaps'),
       col = c('navy', 'red'), lty = c(1, 2), lwd = 2, bty = 'n')


**Year-block permutation result:** the observed maximum ECDF gap is **0.198**. About **47.7%** of the 10,000 shuffled groupings have a gap at least this large; the shuffled 95th percentile is **0.313**. The visible differences among these three periods are therefore not unusual under the exchangeable-year reference scenario.

![Distribution of maximum ECDF gaps after permuting whole years](figures/ipca_monthly_distribution_year_block_permutation.png)


### Interpretation limits

The histograms and rolling quantiles show variation in the observed sample, including a lower median and more negative months in 2018–2025 than in 2010–2017. The year-block permutation check does **not** find the three-period ECDF separation unusual under exchangeable years (tail fraction about 0.48). That result does not establish a fixed inflation distribution: years may be dependent, the period boundaries are choices, and the simulation does not identify causes.

The final notebook window uses data through December 2025. This analysis uses realized monthly IPCA, not Focus forecast errors.
